In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/sample_submission.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/documents.csv
/kaggle/input/competitions/agriculture-climate-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/agriculture-climate-slm-challenge/baseline_submission.csv


In [2]:
import os
import pandas as pd

# To find the file paths dynamically from /kaggle/input
train_path = None
test_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if filename == 'train_qa.csv':
            train_path = full_path
        elif filename == 'test_questions.csv':
            test_path = full_path

print(f"Train path: {train_path}")
print(f"Test path: {test_path}")

# Loading CSV files
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Displaying top of files
print("\n--- Train Head ---")
display(train_df.head())

print("\n--- Test Head ---")
display(test_df.head())

Train path: /kaggle/input/competitions/agriculture-climate-slm-challenge/train_qa.csv
Test path: /kaggle/input/competitions/agriculture-climate-slm-challenge/test_questions.csv

--- Train Head ---


,question,topic,crop,agro_zone,document_id,reference_answer,QuestionId
0,Weevils in stored maize without chemicals?,post_harvest,maize,semi_arid,doc_pos_001,Dry to twelve to thirteen percent and seal in ...,1
1,Insurance paid but my field still failed — why?,climate_adaptation,general,semi_arid,doc_cli_003,Basis risk means index payouts may not match i...,2
2,Which cover crop helps between maize seasons?,soil_health,general,sub_humid,doc_soi_002,Mucuna or lablab reduce erosion and suppress w...,3
3,Maize stalks lodging before harvest — nutrient...,fertiliser,maize,sub_humid,doc_fer_001,Ensure balanced NPK including potassium for st...,4
4,What are signs of bean rust?,crop_diseases,beans,highland,doc_dis_002,Reddish-brown pustules on leaf undersides in c...,5



--- Test Head ---


,QuestionId,question,topic,crop,agro_zone
0,1001,How should I apply nitrogen to leaching-prone ...,crop_diseases,maize,sub_humid
1,1002,Grass gone in August — feed strategy?,livestock,livestock,semi_arid
2,1003,Red spots under my bean leaves during the rains.,crop_diseases,beans,highland
3,1004,Fresh cow dung on vegetable beds — safe?,fertiliser,general,sub_humid
4,1005,How much compost per hectare?,fertiliser,general,sub_humid


In [3]:
# Cell 1: Install rank-bm25
!pip install rank-bm25 -q

In [4]:
# Cell 2: BM25 Retrieval with Metadata Boosting
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

# 1. Tokenize training corpus
tokenized_train = [
    f"{q} {c} {z} {t}".lower().split()
    for q, c, z, t in zip(
        train_df['question'].fillna(''),
        train_df['crop'].fillna(''),
        train_df['agro_zone'].fillna(''),
        train_df['topic'].fillna('')
    )
]

bm25 = BM25Okapi(tokenized_train)

# 2. Score and retrieve best match for each test row
bm25_answers = []
for idx, row in test_df.iterrows():
    query = f"{row['question']} {row.get('crop','')} {row.get('agro_zone','')} {row.get('topic','')}".lower().split()
    scores = np.array(bm25.get_scores(query))
    
    # Apply metadata bonuses to refine ranking
    for j in range(len(train_df)):
        row_train = train_df.iloc[j]
        bonus = 1.0
        if str(row.get('crop', '')).lower() == str(row_train.get('crop', '')).lower() and row.get('crop'):
            bonus += 0.35
        if str(row.get('agro_zone', '')).lower() == str(row_train.get('agro_zone', '')).lower() and row.get('agro_zone'):
            bonus += 0.25
        if str(row.get('topic', '')).lower() == str(row_train.get('topic', '')).lower() and row.get('topic'):
            bonus += 0.20
        scores[j] *= bonus
        
    best_idx = scores.argmax()
    bm25_answers.append(train_df['reference_answer'].iloc[best_idx])

test_df['Answer'] = bm25_answers

# 3. Clean and Export
def clean_answer(text):
    return " ".join(str(text).split()).strip()

test_df['Answer'] = test_df['Answer'].apply(clean_answer)

submission = test_df[['QuestionId', 'Answer']].copy()
submission['Answer'] = submission['Answer'].fillna("No answer provided.")
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("BM25 predictions generated and saved to /kaggle/working/submission.csv")
display(submission.head(12))

BM25 predictions generated and saved to /kaggle/working/submission.csv


,QuestionId,Answer
0,1001,Only if rain or irrigation is imminent; otherw...
1,1002,"Shade, water ad libitum, and avoid working ani..."
2,1003,Reddish-brown pustules on leaf undersides in c...
3,1004,Only if rain or irrigation is imminent; otherw...
4,1005,"It should be dark, crumbly, and free of undeco..."
5,1006,Stem borer tunneling kills the growing point i...
6,1007,Build contour bunds with grass strips and repa...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,When more than ten percent of plants are infes...


In [5]:
# Format submission
submission = test_df[['QuestionId', 'Answer']].copy()
submission['Answer'] = submission['Answer'].fillna("No answer provided.")

# Save directly to /kaggle/working/submission.csv
submission.to_csv("/kaggle/working/submission.csv", index=False)

# Verification checks
print(f"Saved: /kaggle/working/submission.csv")
print(f"Row count: {len(submission)} (expected: 12)")
print(f"Columns: {list(submission.columns)} (expected: ['QuestionId', 'Answer'])")
print(f"Null answers: {submission['Answer'].isna().sum()}")
print(f"Empty answers: {(submission['Answer'].str.strip() == '').sum()}")

display(submission)

Saved: /kaggle/working/submission.csv
Row count: 12 (expected: 12)
Columns: ['QuestionId', 'Answer'] (expected: ['QuestionId', 'Answer'])
Null answers: 0
Empty answers: 0


,QuestionId,Answer
0,1001,Only if rain or irrigation is imminent; otherw...
1,1002,"Shade, water ad libitum, and avoid working ani..."
2,1003,Reddish-brown pustules on leaf undersides in c...
3,1004,Only if rain or irrigation is imminent; otherw...
4,1005,"It should be dark, crumbly, and free of undeco..."
5,1006,Stem borer tunneling kills the growing point i...
6,1007,Build contour bunds with grass strips and repa...
7,1008,Stem borer tunneling kills the growing point i...
8,1009,"It should be dark, crumbly, and free of undeco..."
9,1010,When more than ten percent of plants are infes...
